In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. 加载上一阶段保存的"琥珀" (Pickle文件)
# 这样我们就不用重新跑预处理，直接读取，速度超快
INPUT_FILE = 'processed_data/df_step1_cleaned.pkl'

print("⏳ 正在加载 Phase 1 的数据...")
if os.path.exists(INPUT_FILE):
    df = pd.read_pickle(INPUT_FILE)
    print(f"✅ 加载成功！数据形状: {df.shape}")
else:
    print("❌ 找不到文件，请检查 Phase 1 是否保存成功。")

# 备份一下原始数据，万一改乱了方便恢复
df_backup = df.copy()

⏳ 正在加载 Phase 1 的数据...
✅ 加载成功！数据形状: (3029400, 12)


In [3]:
def create_date_features(df):
    print("⏳ 正在提取时间特征...")
    
    # 确保是时间格式
    df['date'] = pd.to_datetime(df['date'])
    
    # --- 基础时间特征 ---
    # 告诉模型具体的年、月、日、星期
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek  # 0=周一, 6=周日
    
    # 是否是周末 (周六=5, 周日=6)
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(np.int8)
    
    # --- 厄瓜多尔特殊工资日逻辑 (加分项!) ---
    # 规律：每月15日是发薪日，每月最后一天也是发薪日
    df['is_payday'] = 0
    
    # 1. 标记15号
    df.loc[df['day'] == 15, 'is_payday'] = 1
    
    # 2. 标记月末 (因为大小月天数不同，用 is_month_end 属性最准确)
    df.loc[df['date'].dt.is_month_end, 'is_payday'] = 1
    
    print("✅ 时间特征提取完毕！")
    return df

# 执行函数
df = create_date_features(df)

# 看看我们造出来的新列
display(df[['date', 'dayofweek', 'is_weekend', 'is_payday']].head(100))

⏳ 正在提取时间特征...
✅ 时间特征提取完毕！


,date,dayofweek,is_weekend,is_payday
0,2013-01-01,1,0,0
1,2013-01-01,1,0,0
2,2013-01-01,1,0,0
3,2013-01-01,1,0,0
4,2013-01-01,1,0,0
...,...,...,...,...
95,2013-01-01,1,0,0
96,2013-01-01,1,0,0
97,2013-01-01,1,0,0
98,2013-01-01,1,0,0


In [4]:
def create_lag_rolling_features(df):
    print("⏳ 正在排序数据 (这是为了确保时间顺序正确)...")
    # 必须先排序！否则 shift 可能会把张三的数据移给李四
    df = df.sort_values(['store_nbr', 'family', 'date'])
    
    print("⏳ 正在计算滞后特征 (Lags)...")
    # 1. Lag Features (回顾过去)
    # Lag 1: 昨天卖了多少？
    # Lag 7: 上周今天卖了多少？(很重要，因为周末和周末比，周一和周一比)
    lags = [1, 7, 14, 28]
    for lag in lags:
        # groupby保证了我们是在"同一个商店的同一种商品"内部移动时间
        df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)

    print("⏳ 正在计算滑动窗口特征 (Rolling)...")
    # 2. Rolling Features (趋势分析)
    # 我们计算"过去7天"和"过去30天"的平均销量
    # 注意：必须用 shift(1)！因为预测今天时，我们只能看到"昨天及以前"的数据，不能包含今天！
    # min_periods=1 意味着只要有一天有数据就算，不产生空值
    
    # 过去7天均值
    df['rolling_mean_7'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
    )
    
    # 过去30天均值
    df['rolling_mean_30'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(window=30, min_periods=1).mean()
    )
    
    print("✅ 历史特征提取完毕！")
    return df

# 执行函数
df = create_lag_rolling_features(df)

# 再次优化内存 (因为新加了很多 float 列)
# 假设你在 02 文件里还没有定义这个函数，记得把之前那个优化函数 copy 过来，或者这里简单处理
# 如果前面没定义 reduce_mem_usage，这里可以先跳过，或者把之前的函数复制过来运行一次
# df = reduce_mem_usage(df) 

# 看看结果
# 我们选一个具体的商店(1号店)和商品(面包)来看看
mask = (df['store_nbr'] == 1) & (df['family'] == 'BREAD/BAKERY')
cols_to_show = ['date', 'sales', 'sales_lag_1', 'sales_lag_7', 'rolling_mean_7']
display(df[mask][cols_to_show].head(10))

⏳ 正在排序数据 (这是为了确保时间顺序正确)...
⏳ 正在计算滞后特征 (Lags)...
⏳ 正在计算滑动窗口特征 (Rolling)...
✅ 历史特征提取完毕！


,date,sales,sales_lag_1,sales_lag_7,rolling_mean_7
5,2013-01-01,0.000000,NaN,NaN,NaN
1787,2013-01-02,470.652008,0.000000,NaN,0.000000
3569,2013-01-03,310.654999,470.652008,NaN,235.326004
5351,2013-01-04,198.365997,310.654999,NaN,260.435669
7133,2013-01-05,301.057007,198.365997,NaN,244.918251
8915,2013-01-06,147.182007,301.057007,NaN,256.146002
10697,2013-01-07,309.675995,147.182007,NaN,237.985336
12479,2013-01-08,321.851013,309.675995,0.000000,248.226859
14261,2013-01-09,298.582001,321.851013,470.652008,294.205575
16043,2013-01-10,303.515015,298.582001,310.654999,269.624146


In [6]:
def create_zero_sales_features(df):
    print("⏳ 正在计算零销售特征 (遵循文档 1.3)...")
    
    # 1. 标记每一天是否是零销售 (0=有销量, 1=零销量)
    # 注意：这里用的是原始 sales，计算特征时必须 Shift，防止偷看答案
    df['is_zero_sales'] = (df['sales'] == 0).astype(int)
    
    # 确保按时间排序
    df = df.sort_values(['store_nbr', 'family', 'date'])
    
    # --- 特征 A: 历史零销售频率 (Expanding Mean) ---
    # 解释：截止到"昨天"，历史上零销量的概率是多少？
    # shift(1) 是为了保证只看"昨天及以前"
    # expanding().mean() 是计算从开始到现在的累计平均值
    df['zero_sales_freq'] = df.groupby(['store_nbr', 'family'])['is_zero_sales'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    
    # --- 特征 B: 连续零销售天数 (Consecutive Zero Days) ---
    # 解释：直到昨天为止，已经连续多少天没开张了？
    # 这是一个比较高级的计算，逻辑是：计算距离上一次"非零销售"过了多少天
    
    # 1. 标记"有销量"的日子 (sales > 0)
    # 我们把 sales > 0 的位置标记为 True
    # 然后用 cumsum() 生成分组编号。每遇到一次有销量，编号就会 +1
    # 这样，连续的一段 0 就会拥有同一个编号
    sale_groups = (df['sales'] > 0).cumsum()
    
    # 2. 在每个分组内计数
    # 比如：有销量(Grp1) -> 0(Grp1) -> 0(Grp1) -> 有销量(Grp2)
    # 计数：   0            1           2            0
    # 注意：还是要 shift(1)，因为我们预测今天时，只能基于"昨天及以前"的状态
    df['consecutive_zero_days'] = df.groupby(['store_nbr', 'family', sale_groups]).cumcount().shift(1)
    
    # 第一天会产生 NaN，填为 0
    df['consecutive_zero_days'] = df['consecutive_zero_days'].fillna(0)
    df['zero_sales_freq'] = df['zero_sales_freq'].fillna(0)
    
    # 清理掉辅助列，节省内存
    df = df.drop(columns=['is_zero_sales'])
    
    print("✅ 零销售特征 (拼图2.5) 提取完毕！")
    return df

# 执行
df = create_zero_sales_features(df)

# 🕵️‍♂️ 检查一下
# 看看那些本来 sales 是 0 的行，consecutive_zero_days 是不是在增加？
# 我们找一个销量为 0 比较多的商品来看看
mask = (df['store_nbr'] == 1) & (df['family'] == 'BOOKS')
cols = ['date', 'sales', 'consecutive_zero_days', 'zero_sales_freq']
display(df[mask][cols].head(10))

⏳ 正在计算零销售特征 (遵循文档 1.3)...
✅ 零销售特征 (拼图2.5) 提取完毕！


,date,sales,consecutive_zero_days,zero_sales_freq
4,2013-01-01,0.0,16.0,0.0
1786,2013-01-02,0.0,0.0,1.0
3568,2013-01-03,0.0,1.0,1.0
5350,2013-01-04,0.0,2.0,1.0
7132,2013-01-05,0.0,3.0,1.0
8914,2013-01-06,0.0,4.0,1.0
10696,2013-01-07,0.0,5.0,1.0
12478,2013-01-08,0.0,6.0,1.0
14260,2013-01-09,0.0,7.0,1.0
16042,2013-01-10,0.0,8.0,1.0


In [5]:
def create_external_features(df, holidays_df):
    print("⏳ 正在处理外部特征 (节假日 & 油价 & 促销)...")
    
    # --- 1. 节假日处理 (遵循文档 Section 2.5) ---
    # 逻辑：如果 transferred == True，说明这一天假期被调走了，当天是工作日，不算假期
    # 我们只保留 transferred == False 的行
    true_holidays = holidays_df[holidays_df['transferred'] == False].copy()
    
    # 将日期转为 datetime 方便合并
    true_holidays['date'] = pd.to_datetime(true_holidays['date'])
    
    # 只需要日期和类型
    # 注意：同一天可能有多个假期记录（比如一个是地区性，一个是全国性）
    # 我们去重，只要这一天有假，就标记为 1
    holiday_dates = true_holidays['date'].unique()
    
    # 创建 is_holiday 特征
    # isin() 判断当前日期是否在假期列表里
    df['is_holiday'] = df['date'].isin(holiday_dates).astype(np.int8)
    
    # --- 2. 油价高级特征 (遵循文档 Section 2.7) ---
    # 油价已经merge进来了，但我们需要看它的"变化"
    # 计算油价的 7 天移动平均 (平滑波动)
    df['oil_rolling_mean_7'] = df['oil_price'].rolling(window=7, min_periods=1).mean()
    
    # --- 3. 促销特征 (遵循文档 Section 2.6) ---
    # 原始数据里已有 'onpromotion' (即有多少种商品在促销)
    # 我们把 NaN 填为 0 (针对测试集可能出现的空值，虽然通常促销信息是完整的)
    df['onpromotion'] = df['onpromotion'].fillna(0).astype(np.int16)
    
    print("✅ 外部特征 (拼图3) 提取完毕！")
    return df

# 这里的 holidays 是我们在 Phase 1 加载进来的原始数据
# 如果你还没定义 holidays，记得确保它被 read_csv 进来了
df = create_external_features(df, pd.read_csv('raw data/holidays_events.csv'))

# 检查一下结果
# 看看元旦那天 (2013-01-01) 的 is_holiday 是不是 1
print("检查元旦节假日标记:")
display(df[df['date'] == '2013-01-01'][['date', 'is_holiday', 'sales']].head(1))

# 看看油价特征
print("检查油价特征:")
display(df[['date', 'oil_price', 'oil_rolling_mean_7']].tail())

⏳ 正在处理外部特征 (节假日 & 油价 & 促销)...
✅ 外部特征 (拼图3) 提取完毕！
检查元旦节假日标记:


,date,is_holiday,sales
0,2013-01-01,1,0.0


检查油价特征:


/opt/anaconda3/envs/ai6102/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,date,oil_price,oil_rolling_mean_7
3022139,2017-08-27,46.81250,47.486607
3023921,2017-08-28,46.40625,47.348214
3025703,2017-08-29,46.46875,47.178571
3027485,2017-08-30,45.96875,46.825893
3029267,2017-08-31,47.25000,46.825893


In [8]:
from sklearn.preprocessing import LabelEncoder
import gc
import os

def final_formatting_and_split(df):
    print("⏳ 正在进行最终格式化...")
    
    # --- 🚑 修复 1: 处理 'type' 撞车问题 ---
    # 检查是否存在因为合并产生的 type_x 和 type_y
    if 'type_y' in df.columns:
        print("   🔧 发现列名冲突，正在将 'type_y' 重命名为 'store_type'...")
        df.rename(columns={'type_y': 'store_type'}, inplace=True)
    
    # 如果 type_x 存在，它其实就是我们之前标记的 is_train 辅助信息
    # 我们可以保留它，或者如果已经有 type_x，就不用管它，后面 drop 列表里处理
    
    # --- 🚑 修复 2: 修正编码列表 ---
    # 我们只编码实际存在的列。
    # 删掉了 'locale', 'transferred' 等，因为我们在拼图3里只取了 is_holiday 标记，没有把详细文字 merge 进来
    categorical_cols = ['family', 'city', 'state', 'store_type']
    
    # 填补空值 (防止报错)
    for col in categorical_cols:
        # 检查列是否存在，存在才处理
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').astype(str)
            
            # 编码
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            print(f"   ✅ 已编码: {col}")
        else:
            print(f"   ⚠️ 警告: 找不到列 {col}，跳过编码。")
        
    # --- 3. 处理空值 (Lag特征产生的NaN) ---
    print("⏳ 正在处理历史特征的空值...")
    lag_cols = [c for c in df.columns if 'lag' in c or 'rolling' in c]
    df[lag_cols] = df[lag_cols].fillna(0)
    
    # --- 4. 拆分训练集和测试集 ---
    print("⏳ 正在拆分训练集和测试集...")
    
    # 训练集：时间 < 2017-08-16
    train_df = df[df['date'] < '2017-08-16']
    # 测试集：时间 >= 2017-08-16
    test_df = df[df['date'] >= '2017-08-16']
    
    # 定义要扔掉的列
    # 注意：这里要把 type_x (如果有的话) 也扔掉，它是原来的 train/test 标记
    cols_to_drop = ['date', 'id', 'sales', 'is_train', 'type_x', 'type']
    
    # 只 drop 实际存在的列
    existing_drop_cols = [c for c in cols_to_drop if c in df.columns]
    
    X_train = train_df.drop(columns=existing_drop_cols)
    y_train = train_df['sales']
    X_test = test_df.drop(columns=existing_drop_cols)
    
    # --- 5. 保存文件 ---
    if not os.path.exists('processed_data'):
        os.makedirs('processed_data')
        
    print(f"📦 打包中... (X_train shape: {X_train.shape})")
    X_train.to_pickle('processed_data/X_train.pkl')
    y_train.to_pickle('processed_data/y_train.pkl')
    X_test.to_pickle('processed_data/X_test.pkl')
    
    print("\n🎉🎉🎉 Phase 2 完美收官！(已修复列名冲突问题)")
    return X_train, y_train, X_test

# 执行
X_train, y_train, X_test = final_formatting_and_split(df)

⏳ 正在进行最终格式化...
   🔧 发现列名冲突，正在将 'type_y' 重命名为 'store_type'...
   ✅ 已编码: family
   ✅ 已编码: city
   ✅ 已编码: state
   ✅ 已编码: store_type
⏳ 正在处理历史特征的空值...
⏳ 正在拆分训练集和测试集...
📦 打包中... (X_train shape: (3000888, 24))

🎉🎉🎉 Phase 2 完美收官！(已修复列名冲突问题)
